<a href="https://colab.research.google.com/github/lsteffenel/M2Atmo_et_Climat/blob/main/12-PINN_RNET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP — Physics-Informed Neural Networks (PINN)
## Application à la diffusion de la chaleur dans le sol

**Master 2 RNET — Météorologie** · Objectif : découvrir les PINN à partir de vos acquis en deep learning avec Keras.

---

### 1. Pourquoi des PINN ?

Jusqu'ici, vous avez entraîné des réseaux de neurones **à partir de données étiquetées** (classification, transfer learning, séries temporelles). Le réseau apprend une correspondance entrée → sortie à partir d'exemples.

Un **Physics-Informed Neural Network** (Raissi, Perdikaris & Karniadakis, 2019) ajoute une seconde source d'information : **la physique elle-même**, sous forme d'équations aux dérivées partielles (EDP). L'idée est simple :

1. Le réseau approxime la solution : $T(z, t) \mapsto T_\theta(z, t)$.
2. Grâce à la **dérivation automatique** (autodiff), on peut calculer les dérivées $\partial T_\theta / \partial t$, $\partial T_\theta / \partial z$, $\partial^2 T_\theta / \partial z^2$... même si le réseau est profond et non-linéaire.
3. On définit une **fonction de perte physique** : la résidu de l'EDP évalué en des points de collocation aléatoires :
$$\mathcal{L}_{phys} = \frac{1}{N_c} \sum_i \left( \frac{\partial T_\theta}{\partial t}(z_i, t_i) - \kappa \frac{\partial^2 T_\theta}{\partial z^2}(z_i, t_i) \right)^2$$
4. On ajoute les **conditions aux limites et initiales** (et éventuellement des données de mesure) :
$$\mathcal{L} = \lambda_{phys}\,\mathcal{L}_{phys} + \lambda_{CL}\,\mathcal{L}_{CL} + \lambda_{CI}\,\mathcal{L}_{CI}$$

On force ainsi le réseau à être une solution de l'EDP, pas seulement à interpoler des données. Intérêt en météo/océanographie : assimilation de données clairsemées, super-résolution, modèles hybrides physique + ML.

### 2. Le problème étudié : onde thermique diurne dans le sol

Un problème classique de micrométéorologie : la température de surface oscille avec un cycle diurne, et cette onde thermique **pénètre dans le sol** en s'amortissant et en se déphasant, selon l'équation de diffusion :

$$\frac{\partial T}{\partial t} = \kappa \frac{\partial^2 T}{\partial z^2}, \qquad z \in [0, L],\; t \in [0, t_{max}]$$

où $z$ est la profondeur (vers le bas), $\kappa$ la diffusivité thermique du sol ($\approx 5\times 10^{-7}$ m²/s pour un sol sec).

Pour une surface périodique $T(0,t) = \bar{T} + A \sin(\omega t)$, la solution analytique du milieu semi-infini est connue :

$$T(z, t) = \bar{T} + A\, e^{-z/d} \sin(\omega t - z/d), \qquad d = \sqrt{\frac{2\kappa}{\omega}}$$

C'est notre **solution de référence** pour valider le PINN. L'onde s'amortit avec la profondeur et se déphase — le PINN devra reproduire ces deux effets sans jamais "voir" la solution analytique, seulement l'EDP + la température de surface.

### 3. Organisation du TP

- **Partie A** : code complet, à exécuter et comprendre (Chapitres 1 à 5).
- **Partie B** : extension — un second problème (advection de température) dont une partie du code est **à compléter** (Chapitre 6), suivi de questions d'analyse.

**Prérequis techniques** : TensorFlow/Keras (déjà installé sur Colab). Nous n'utilisons pas `.fit()` mais une boucle d'entraînement personnalisée avec `tf.GradientTape`, car nous devons dériver la perte par rapport **aux entrées** du réseau, ce que `.fit()` ne permet pas.

## Chapitre 0 — Environnement

Aucune installation nécessaire sur Colab (TensorFlow 2.x est préinstallé). Nous fixons aussi une graine aléatoire pour la reproductibilité.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow :", tf.__version__)

# Petite fonction utilitaire pour afficher les courbes de perte
def plot_losses(loss_history):
    plt.figure(figsize=(7, 4))
    for key, values in loss_history.items():
        plt.plot(values, label=key)
    plt.yscale('log')
    plt.xlabel('Époque')
    plt.ylabel('Perte (log)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title('Évolution des pertes')
    plt.show()

## Chapitre 1 — Solution de référence

Nous fixons les paramètres physiques, puis nous écrivons la solution analytique. Elle servira uniquement à **valider** le PINN (et à visualiser le phénomène), jamais à l'entraîner.

Attention aux unités : nous travaillons en secondes et mètres, avec une profondeur maximale $L = 0.4$ m et une durée $t_{max} = 48$ h (deux cycles diurnes, pour que la solution analytique transitoire soit bien établie).

In [ ]:
# --- Paramètres physiques ---
kappa = 5e-7          # diffusivité thermique du sol [m2/s]
T_bar  = 20.0         # température moyenne de surface [°C]
A      = 8.0          # amplitude du cycle diurne [°C]
omega  = 2*np.pi / (24*3600)   # pulsation diurne [rad/s]

L      = 0.4          # profondeur maximale [m]
T_MAX  = 48*3600      # durée de simulation [s]

d = np.sqrt(2*kappa/omega)     # profondeur d'amortissement ( skin depth )
print(f"Profondeur de pénétration de l'onde diurne : d = {d*100:.1f} cm")

# --- Solution analytique du milieu semi-infini ---
def T_exact(z, t):
    """z en m, t en s. Convention : z=0 surface, z croît vers le bas."""
    z = np.atleast_1d(z).astype(np.float64)
    t = np.atleast_1d(t).astype(np.float64)
    return T_bar + A * np.exp(-z/d) * np.sin(omega*t - z/d)

In [ ]:
# --- Visualisation de la solution de référence ---
z_plot = np.linspace(0, L, 100)
t_hours = np.linspace(0, T_MAX, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Profils verticaux à différents instants
for h in [6, 12, 18]:
    T_prof = T_exact(z_plot, h*3600)
    axes[0].plot(T_prof, -z_plot, label=f"t = {h} h")
axes[0].set_xlabel('T (°C)')
axes[0].set_ylabel('Profondeur z (m) [axe inversé]')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Profils de température dans le sol')

# Cycle de température à différentes profondeurs
for z0 in [0.0, 0.10, 0.20]:
    axes[1].plot(t_hours/3600, T_exact(np.full_like(t_hours, z0), t_hours).squeeze(),
                 label=f"z = {z0*100:.0f} cm")
axes[1].set_xlabel('Temps (h)')
axes[1].set_ylabel('T (°C)')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('Cycle diurne amorti et déphasé avec la profondeur')
plt.tight_layout(); plt.show()

## Chapitre 2 — Le PINN : architecture et données d'entraînement

### 2.1 Le réseau

Le réseau prend en entrée le couple $(z, t)$ et sort $T$. Pour un PINN, l'activation `tanh` est fortement recommandée : elle est infiniment dérivable (le calcul du résidu exige $\partial^2 T/\partial z^2$), contrairement à `relu` dont la dérivée seconde est discontinue. Un réseau de 4 couches de 32 neurones suffit largement ici.

**Point clé** : nous normalisons les entrées et sorties. Les pertes physiques contiennent des dérivées secondes ; si $z$ et $t$ varient sur des échelles très différentes ($0.4$ m vs $172800$ s), les dérivées $\partial/\partial t$ sont minuscules et l'entraînement diverge. Nous travaillons donc en variables adimensionnées $\tilde{z} = z/L \in [0,1]$, $\tilde{t} = t/t_{max} \in [0,1]$ et $\tilde{T} = (T - \bar{T})/A$. La chaîne de dérivation (chain rule) est gérée par l'autodiff.

### 2.2 Les trois ensembles de points

1. **Points de collocation** $(z_i, t_i)$ : tirés uniformément dans le domaine. Le résidu de l'EDP y est pénalisé. Pas de valeur cible : c'est la physique qui contraint.
2. **Points de surface** : $z = 0$, avec la valeur imposée $T = \bar{T} + A\sin(\omega t)$ (condition aux limites de Dirichlet — ici elle est analytique, mais elle pourrait venir de mesures).
3. **Condition initiale** : à $t=0$, la solution analytique du semi-infini est $T(z, 0) = \bar T + A e^{-z/d}\sin(-z/d)$. Pour simplifier (cas usuel en forçage périodique établi), nous fournissons ces valeurs comme CI ; en pratique, elles proviendraient d'un profil mesuré ou d'un modèle.

In [ ]:
# --- Normalisation / adimensionnement ---
# Tilde : z/L, t/T_MAX, (T - T_bar)/A
def to_phys(zT, tT, TT):   # adimensionné -> physique
    return zT*L, tT*T_MAX, T_bar + TT*A

def to_norm(z, t, T=None):  # physique -> adimensionné
    zT, tT = z/L, t/T_MAX
    TT = None if T is None else (T - T_bar)/A
    return zT, tT, TT

In [ ]:
N_col = 4000   # points de collocation
N_bc  = 400    # points de condition aux limites (surface)
N_ic  = 400    # points de condition initiale

def sample_training_points():
    # 1) Collocation : uniforme dans [0,1] x [0,1] (variables adimensionnées)
    z_c = tf.random.uniform((N_col, 1))
    t_c = tf.random.uniform((N_col, 1))

    # 2) Surface : z=0, t uniforme, T imposé par le forçage diurne
    t_b = tf.random.uniform((N_bc, 1))
    T_b = (np.sin(omega * t_b.numpy() * T_MAX)).astype(np.float32)  # (T-T_bar)/A
    z_b = tf.zeros((N_bc, 1))

    # 3) Condition initiale : t=0, profil issu de la solution semi-infinie
    z_i = tf.random.uniform((N_ic, 1))
    T_i = (np.exp(-z_i.numpy()*L/d) * np.sin(-z_i.numpy()*L/d)).astype(np.float32)
    t_i = tf.zeros((N_ic, 1))

    return (z_c, t_c), (z_b, t_b, T_b), (z_i, t_i, T_i)

(z_c, t_c), (z_b, t_b, T_b), (z_i, t_i, T_i) = sample_training_points()
print("Collocation :", z_c.shape, "| Surface :", z_b.shape, "| Initiale :", z_i.shape)

### 2.3 Le résidu physique avec autodiff

C'est le cœur du PINN. Dans un `tf.GradientTape`, on enregistre les entrées `z, t`, on évalue le réseau, puis :

- `tape.gradient(T, t)` → $\partial T/\partial t$
- dérivée première en $z$ puis seconde via deux `GradientTape` imbriqués → $\partial^2 T/\partial z^2$

Le réseau prend en entrée les variables adimensionnées $(\tilde z, \tilde t)$ : ses dérivées (calculées par autodiff) sont donc $\partial \tilde T/\partial \tilde t$ et $\partial^2 \tilde T/\partial \tilde z^2$. En réécrivant l'EDP avec la règle de dérivation en chaîne, on obtient une **EDP adimensionnée** :

$$\frac{\partial \tilde T}{\partial \tilde t} = \alpha\, \frac{\partial^2 \tilde T}{\partial \tilde z^2}, \qquad \alpha = \frac{\kappa\, t_{max}}{L^2}$$

**Pourquoi adimensionner ?** Ici $\alpha \approx 0.54$ : les deux termes du résidu sont d'ordre de grandeur comparable. En unités brutes (secondes, mètres), les dérivées seraient de l'ordre de $10^{-6}$ et $10^{-1}$ respectivement : la perte serait totalement dominée par un seul terme et l'entraînement très difficile. L'adimensionnement est l'étape la plus importante (et la plus souvent négligée) d'un PINN.

In [ ]:
TM = T_MAX          # alias lisible
ALPHA = kappa * T_MAX / L**2   # coefficient de l'EDP adimensionnée
print(f"alpha = {ALPHA:.3f}")

def pde_residual(model, z, t):
    """Résidu de l'EDP adimensionnée : dT/dt - alpha * d2T/dz2
    (T, z, t = variables adimensionnées vues par le réseau)."""
    with tf.GradientTape(persistent=True) as tape2:
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch(z); tape1.watch(t)
            tape2.watch(z); tape2.watch(t)
            T = model(tf.concat([z, t], axis=1))
        dT_dt = tape1.gradient(T, t)          # dT/dt
        dT_dz = tape1.gradient(T, z)          # dT/dz (utile pour l'extension)
    d2T_dz2 = tape2.gradient(dT_dz, z)        # d2T/dz2
    del tape1, tape2                          # libère la mémoire du graphe
    return dT_dt - ALPHA * d2T_dz2

### 2.4 Construction du réseau (Keras)

Architecture minimale. Le modèle est un `tf.keras.Sequential` classique : tout ce que vous savez de Keras reste valable, seule la boucle d'entraînement change.

In [ ]:
def build_pinn(width=32, depth=4):
    layers = [tf.keras.layers.Input(shape=(2,))]
    for _ in range(depth):
        layers.append(tf.keras.layers.Dense(width, activation='tanh',
                                            kernel_initializer='glorot_normal'))
    layers.append(tf.keras.layers.Dense(1))   # sortie linéaire
    return tf.keras.Sequential(layers)

model = build_pinn()
model.summary()

## Chapitre 3 — Boucle d'entraînement personnalisée

Trois étapes par itération, répétées à la manière d'un `tf.GradientTape` "from scratch" :

1. Calculer la perte totale $\mathcal{L} = w_{phys}\mathcal{L}_{phys} + w_{CL}\mathcal{L}_{CL} + w_{CI}\mathcal{L}_{CI}$ ;
2. Calculer $\nabla_\theta \mathcal{L}$ (c'est ici que l'autodiff s'applique aux **poids**, comme d'habitude) ;
3. Appliquer la mise à jour via un optimiseur (`Adam`).

**Pondération** : les trois pertes n'ont pas le même ordre de grandeur (le résidu physique en °C/s est très petit). Les poids $w$ ci-dessous compensent cela. L'équilibre des pertes (*loss balancing*) est l'un des sujets de recherche actifs des PINN — vous expérimenterez dessus en Partie B.

In [ ]:
EPOCHS = 2000
w_phys, w_bc, w_ic = 1.0, 10.0, 10.0

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history = {'total': [], 'phys': [], 'bc': [], 'ic': []}

@tf.function
def train_step(z_c, t_c, z_b, t_b, T_b, z_i, t_i, T_i):
    with tf.GradientTape() as tape:
        # --- 1) Perte physique (résidu sur les points de collocation) ---
        r = pde_residual(model, z_c, t_c)
        loss_phys = tf.reduce_mean(tf.square(r))

        # --- 2) Perte de condition aux limites (surface) ---
        T_pred_b = model(tf.concat([z_b, t_b], axis=1))
        loss_bc  = tf.reduce_mean(tf.square(T_pred_b - T_b))

        # --- 3) Perte de condition initiale ---
        T_pred_i = model(tf.concat([z_i, t_i], axis=1))
        loss_ic  = tf.reduce_mean(tf.square(T_pred_i - T_i))

        # --- Perte totale pondérée ---
        loss = w_phys*loss_phys + w_bc*loss_bc + w_ic*loss_ic

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss, loss_phys, loss_bc, loss_ic

import time
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    loss, lp, lb, li = train_step(z_c, t_c, z_b, t_b, T_b, z_i, t_i, T_i)
    loss_history['total'].append(float(loss))
    loss_history['phys'].append(float(lp))
    loss_history['bc'].append(float(lb))
    loss_history['ic'].append(float(li))
    if epoch % 400 == 0:
        print(f"Époque {epoch:5d} | total={float(loss):.2e} | "
              f"phys={float(lp):.2e} | bc={float(lb):.2e} | ic={float(li):.2e}")

print(f"Entraînement terminé en {time.time()-t0:.0f} s")

## Chapitre 4 — Validation contre la solution analytique

Le moment de vérité : le réseau n'a jamais vu la solution analytique, seulement l'EDP, le forçage de surface et le profil initial. Comparons.

In [ ]:
def predict_T(z, t):
  """Prédiction en unités physiques. z, t : scalaires ou arrays numpy."""
  z = np.atleast_1d(np.asarray(z, dtype=np.float32))
  t = np.atleast_1d(np.asarray(t, dtype=np.float32))
  # Diffusion (broadcast) : si l'un des deux est un scalaire, on le
  # réplique à la taille del'autre (ex. un profil à un instant donné).
  if z.size == 1 and t.size > 1:
    z = np.full_like(t, z[0])
  if t.size == 1 and z.size > 1:
    t = np.full_like(z, t[0])
  zt = np.hstack([z.reshape(-1, 1)/L, t.reshape(-1, 1)/TM])
  TT = model(zt).numpy().squeeze()
  return T_bar + A*TT

# --- Grille de validation (profondeur x temps) ---
z_grid = np.linspace(0, L, 60)
t_grid = np.linspace(0, T_MAX, 120)
ZZ, TT_grid = np.meshgrid(z_grid, t_grid)

T_true = T_exact(ZZ.ravel(), TT_grid.ravel()).reshape(ZZ.shape)
T_pred = predict_T(ZZ.ravel(), TT_grid.ravel()).reshape(ZZ.shape)

err = T_pred - T_true
print(f"Erreur absolue moyenne : {np.mean(np.abs(err)):.3f} °C")
print(f"Erreur absolue max     : {np.max(np.abs(err)):.3f} °C")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
im0 = axes[0].pcolormesh(t_grid/3600, -z_grid*100, T_true.T, shading='auto', cmap='coolwarm')
axes[0].set_title('Solution analytique')
axes[0].set_xlabel('Temps (h)'); axes[0].set_ylabel('Profondeur (cm)')
plt.colorbar(im0, ax=axes[0], label='T (°C)')

im1 = axes[1].pcolormesh(t_grid/3600, -z_grid*100, T_pred.T, shading='auto', cmap='coolwarm')
axes[1].set_title('Prédiction du PINN')
axes[1].set_xlabel('Temps (h)')
plt.colorbar(im1, ax=axes[1], label='T (°C)')

im2 = axes[2].pcolormesh(t_grid/3600, -z_grid*100, err.T, shading='auto', cmap='viridis')
axes[2].set_title('Erreur (PINN - analytique)')
axes[2].set_xlabel('Temps (h)')
plt.colorbar(im2, ax=axes[2], label='ΔT (°C)')
plt.tight_layout(); plt.show()

In [ ]:
# --- Comparaison des profils et des cycles ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for h in [6, 12, 18]:
    axes[0].plot(T_exact(z_grid, h*3600), -z_grid*100, 'k-', alpha=0.6, label=f'Analytique t={h}h')
    axes[0].plot(predict_T(z_grid, h*3600), -z_grid*100, 'r--', alpha=0.8)
axes[0].set_xlabel('T (°C)'); axes[0].set_ylabel('Profondeur (cm)')
axes[0].set_title('Profils : analytique (trait plein) vs PINN (pointillé)')
axes[0].legend(); axes[0].grid(alpha=0.3)

for z0 in [0.0, 0.10, 0.20]:
    axes[1].plot(t_grid/3600, T_exact(np.full_like(t_grid, z0), t_grid), 'k-', alpha=0.6)
    axes[1].plot(t_grid/3600, predict_T(np.full_like(t_grid, z0), t_grid), 'r--', alpha=0.8)
axes[1].set_xlabel('Temps (h)'); axes[1].set_ylabel('T (°C)')
axes[1].set_title('Cycles à z=0, 10, 20 cm : analytique vs PINN')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

plot_losses(loss_history)

## Chapitre 5 — Analyse du comportement

Quelques observations à discuter (voir questions en Partie B) :

- L'erreur est généralement plus grande près de la surface et aux bords temporels ($t=0$ et $t=t_{max}$), où les contraintes changent de nature.
- Le résidu physique est évalué en $\sim$4000 points seulement : c'est une discrétisation stochastique, bien différente d'un schéma aux différences finies.
- Contrairement à un schéma numérique, le PINN donne une solution **continue** $T(z,t)$ évaluable partout, sans grille — mais sans garantie de stabilité ni de conservation.
- L'entraînement est plus coûteux qu'une différence finie explicite pour ce petit problème 1D ; l'intérêt des PINN apparaît en dimension supérieure, en présence de données d'observation clairsemées, ou pour des modèles hybrides.

# Partie B — Extension : advection de température (code à compléter)

## Chapitre 6 — Votre tour

Un second phénomène fondamental en météo : le **transport advectif**. Une perturbation de température se déplace avec le vent. En 1D, avec un vent zonale constant $u > 0$ et une diffusion négligeable :

$$\frac{\partial T}{\partial t} + u\,\frac{\partial T}{\partial x} = 0, \qquad x \in [0, L_x],\; t \in [0, t_{max}]$$

Condition initiale et solution analytique exacte (transport pur) :

$$T(x, t) = T_0(x - u\,t)$$

Prenons un paquet thermique gaussien : $T(x,t) = \exp\left(-\frac{(x - x_0 - u t)^2}{2\sigma^2}\right)$ — la bosse se translate à vitesse $u$ sans se déformer.

**Remarque météo** : l'équation d'advection pure est difficile pour les schémas numériques classiques (diffusion numérique, oscillations) — un bon terrain de test pour un PINN.

Vous disposez ci-dessous d'un squelette. Les blocs `# TODO` sont à compléter ; les indices sont donnés sous chaque bloc. Les cellules de test vérifient vos réponses.


In [ ]:
# --- Paramètres du problème d'advection ---
import numpy as np
import tensorflow as tf

U     = 1.0        # vent [m/s]
LX    = 3.0        # longueur du domaine [m]
TMAX  = 2.0        # durée [s]
X0    = 0.5        # position initiale de la bosse [m]
SIGMA = 0.2        # largeur de la bosse [m]

def T_adv_exact(x, t):
    """Solution analytique : gaussienne translatée."""
    return np.exp(-((x - X0 - U*t)**2) / (2*SIGMA**2))

x_plot = np.linspace(0, LX, 300)
plt.figure(figsize=(8, 3))
for t in [0.0, 0.5, 1.0, 1.5]:
    plt.plot(x_plot, T_adv_exact(x_plot, t), label=f"t = {t} s")
plt.xlabel('x (m)'); plt.ylabel('T (adimensionné)')
plt.title('Solution analytique : advection de la bosse')
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 6.1 Échantillonnage des points d'entraînement

Contrairement au problème de diffusion, la **condition initiale suffit** ici (pas de condition aux limites de surface). On impose en plus une condition aux limites d'entrée $T(0, t) = 0$ (le flux entrant est nul car la bosse n'atteint pas $x=0$).

In [ ]:
N_col = 4000
N_ic  = 400
N_bc  = 200

def sample_adv_points():
    # TODO 1 : points de collocation (x, t) uniformes dans [0,LX] x [0,TMAX]
    # (variables adimensionnées x/LX, t/TMAX, comme au Chapitre 2)
    x_c = ...   # forme (N_col, 1), tf.random.uniform
    t_c = ...

    # TODO 2 : condition initiale : x uniforme, t=0,
    #          cible = gaussienne analytique à t=0
    x_i = ...
    t_i = ...
    T_i = ...   # tf.constant de T_adv_exact(x_physique, 0)

    # TODO 3 : bord d'entrée x=0, t uniforme, cible T=0
    x_b = ...
    t_b = ...
    T_b = ...
    return (x_c, t_c), (x_i, t_i, T_i), (x_b, t_b, T_b)

(x_c, t_c), (x_i, t_i, T_i), (x_b, t_b, T_b) = sample_adv_points()

# --- Vérification rapide ---
assert x_c.shape == (N_col, 1) and t_c.shape == (N_col, 1)
assert float(tf.reduce_max(tf.abs(T_i - 0.0))) >= 0.9  # la bosse atteint ~1
print("Échantillonnage OK")

### 6.2 Résidu de l'EDP d'advection

Complétez le résidu. Comme au Chapitre 2.3, le réseau voit les variables adimensionnées $(\tilde x, \tilde t)$ : en réécrivant l'EDP d'advection avec la règle de dérivation en chaîne, on obtient l'**EDP adimensionnée** :

$$\frac{\partial T}{\partial \tilde t} + \beta\, \frac{\partial T}{\partial \tilde x} = 0, \qquad \beta = \frac{u\, t_{max}}{L_x}$$

Vérifiez l'ordre de grandeur de $\beta$ ci-dessous, puis complétez le résidu $\partial T/\partial \tilde t + \beta\, \partial T/\partial \tilde x$.

In [ ]:
BETA = U * TMAX / LX   # coefficient de l'EDP adimensionnée
print(f"beta = {BETA:.3f}")

def pde_residual_adv(model, x, t):
    """Résidu adimensionné : dT/dt + beta * dT/dx."""
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x); tape.watch(t)
        T = model(tf.concat([x, t], axis=1))
    # TODO 4 : calculer dT/dt et dT/dx avec tape.gradient
    dT_dt = ...
    dT_dx = ...
    del tape
    # TODO 5 : retourner le résidu dT_dt + BETA * dT_dx
    return ...

# --- Test automatique du résidu sur un réseau non entraîné ---
# (vérifie seulement que la fonction tourne et renvoie la bonne forme)
model_adv = build_pinn()
r_test = pde_residual_adv(model_adv, x_c[:100], t_c[:100])
assert r_test.shape == (100, 1), f"Forme incorrecte : {r_test.shape}"
print("Résidu OK")

### 6.3 Boucle d'entraînement

Complétez la fonction de perte puis la boucle. Vous pouvez recopier la structure du Chapitre 3 en adaptant les trois termes : physique (collocation), condition initiale, bord d'entrée.

In [ ]:
EPOCHS_ADV = 3000
w_phys, w_ic, w_bc = 1.0, 30.0, 10.0

optimizer_adv = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_hist_adv = {'total': [], 'phys': [], 'ic': [], 'bc': []}

@tf.function
def train_step_adv(x_c, t_c, x_i, t_i, T_i, x_b, t_b, T_b):
    with tf.GradientTape() as tape:
        # TODO 6 : perte physique = moyenne du carré du résidu
        loss_phys = ...

        # TODO 7 : perte condition initiale = MSE(model([x_i,t_i]) - T_i)
        loss_ic = ...

        # TODO 8 : perte bord d'entrée = MSE(model([x_b,t_b]) - T_b)
        loss_bc = ...

        loss = w_phys*loss_phys + w_ic*loss_ic + w_bc*loss_bc
    grads = tape.gradient(loss, model_adv.trainable_variables)
    optimizer_adv.apply_gradients(zip(grads, model_adv.trainable_variables))
    return loss, loss_phys, loss_ic, loss_bc

for epoch in range(1, EPOCHS_ADV + 1):
    loss, lp, li, lb = train_step_adv(x_c, t_c, x_i, t_i, T_i, x_b, t_b, T_b)
    loss_hist_adv['total'].append(float(loss))
    loss_hist_adv['phys'].append(float(lp))
    loss_hist_adv['ic'].append(float(li))
    loss_hist_adv['bc'].append(float(lb))
    if epoch % 500 == 0:
        print(f"Époque {epoch:5d} | total={float(loss):.2e} | "
              f"phys={float(lp):.2e} | ic={float(li):.2e} | bc={float(lb):.2e}")

In [ ]:
# --- Validation : la bosse doit se déplacer sans se déformer ---
x_grid = np.linspace(0, LX, 200)
t_grid = np.linspace(0, TMAX, 100)
XX, TTg = np.meshgrid(x_grid, t_grid)

T_true_adv = T_adv_exact(XX.ravel(), TTg.ravel()).reshape(XX.shape)
xT = (XX.ravel()/LX).astype(np.float32).reshape(-1, 1)
tT = (TTg.ravel()/TMAX).astype(np.float32).reshape(-1, 1)
T_pred_adv = model_adv(np.hstack([xT, tT])).numpy().reshape(XX.shape)

err = T_pred_adv - T_true_adv
print(f"Erreur absolue moyenne : {np.mean(np.abs(err)):.4f}")
print(f"Erreur absolue max     : {np.max(np.abs(err)):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
im0 = axes[0].pcolormesh(t_grid, x_grid, T_true_adv.T, shading='auto', cmap='coolwarm')
axes[0].set_title('Analytique'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].pcolormesh(t_grid, x_grid, T_pred_adv.T, shading='auto', cmap='coolwarm')
axes[1].set_title('PINN'); plt.colorbar(im1, ax=axes[1])
im2 = axes[2].pcolormesh(t_grid, x_grid, err.T, shading='auto', cmap='viridis')
axes[2].set_title('Erreur'); plt.colorbar(im2, ax=axes[2])
for ax in axes:
    ax.set_xlabel('t (s)')
axes[0].set_ylabel('x (m)')
plt.tight_layout(); plt.show()

plot_losses(loss_hist_adv)

## Questions d'analyse (Partie B)

1. **Résidu** : vérifiez à la main, sur papier, que la solution analytique $T = \exp(-\frac{(x-x_0-ut)^2}{2\sigma^2})$ annule bien le résidu $\partial T/\partial t + u\,\partial T/\partial x$.

2. **Équilibre des pertes** : faites varier `w_phys` de $10^{-2}$ à $10^{2}$ (échelle logarithmique, 5 essais). Tracez l'erreur finale contre `w_phys`. Que se passe-t-il quand la physique est sous-pondérée ? Trop pondérée ?

3. **Densité de collocation** : réentraînez avec `N_col` = 500, 2000, 8000. L'erreur finale dépend-elle linéairement du nombre de points ?

4. **Architecture** : remplacez `tanh` par `relu` dans `build_pinn`. Expliquez pourquoi le résultat se dégrade, en vous appuyant sur la discontinuité de la dérivée seconde de ReLU.

5. **Apprentissage du paramètre physique (bonus)** : supposez que $\kappa$ du sol est inconnu. Transformez `kappa` en `tf.Variable` entraînable (ajoutez-le à la liste des variables de l'optimiseur) et regardez si le PINN peut l'identifier à partir de la seule condition de surface. C'est l'exemple le plus simple d'**assimilation de données par PINN** — exactement le type d'application visé en météorologie.

6. **Discussion météo** : citez deux situations météorologiques où un PINN pourrait rivaliser avec un schéma numérique classique, et deux limitations qui freinent son usage opérationnel (indices : coût d'entraînement, garantie de conservation, multimédium d'échelles, dérivées hautes fréquences — *spectral bias*).

## Références

- Raissi, Perdikaris & Karniadakis (2019), *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear PDEs*, J. Comput. Phys., 378, 686–707.
- Karniadakis et al. (2021), *Physics-informed machine learning*, Nature Reviews Physics, 3, 422–440.
- Cucco & Nofi (2024), un exemple d'application météorologique des PINN aux mécanismes de convection (*Artificial Intelligence for the Earth Systems*, 3, 230042) — à vérifier avant de le citer en copie étudiante